In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from msa.utils.paths import get_gdelt_with_sentiment, get_joined_dataset

MAG7 = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA']

# Load FinBERT-scored articles
df = pd.read_parquet(get_gdelt_with_sentiment("parquet"))
df["seendate"] = pd.to_datetime(df["seendate"])

# Load joined dataset (articles + next-day OHLCV)
join = pd.read_parquet(get_joined_dataset("parquet"))
join["article_date"] = pd.to_datetime(join["article_date"])
join["price_date"] = pd.to_datetime(join["price_date"])

print(f"Articles: {len(df):,} rows | {df['seendate'].min().date()} -> {df['seendate'].max().date()}")
print(f"Join table: {len(join):,} rows")
print(f"Unique domains: {df['domain'].nunique():,}")


Articles: 62,930 rows | 2024-02-08 -> 2026-03-04
Join table: 62,885 rows
Unique domains: 5,126


In [2]:
# Domain volume distribution
domain_counts = df["domain"].value_counts()
domain_pct = domain_counts / len(df) * 100

top10 = pd.DataFrame({
    "articles": domain_counts.head(10),
    "pct": domain_pct.head(10).round(2)
})

print("Top 10 domains by article count:")
print(top10.to_string())
print(f"\nTop 5 share:  {domain_pct.head(5).sum():.1f}%")
print(f"Top 10 share: {domain_pct.head(10).sum():.1f}%")
print(f"Single-article domains: {(domain_counts == 1).sum():,} of {len(domain_counts):,}")


Top 10 domains by article count:
                              articles   pct
domain                                      
yahoo.com                         2539  4.03
fool.com                          2330  3.70
benzinga.com                      1800  2.86
indiatimes.com                    1314  2.09
marketscreener.com                1232  1.96
finance.yahoo.com                 1195  1.90
insidermonkey.com                  907  1.44
webpronews.com                     827  1.31
themarketsdaily.com                684  1.09
markets.financialcontent.com       643  1.02

Top 5 share:  14.6%
Top 10 share: 21.4%
Single-article domains: 1,898 of 5,126


In [3]:
# Sentiment bias by domain (top 20 by volume)
domain_sentiment = (
    df[df["domain"].isin(domain_counts.head(20).index)]
    .groupby("domain")["sentiment_score"]
    .agg(mean="mean", median="median", count="count", std="std")
    .sort_values("count", ascending=False)
    .round(3)
)
print("Sentiment profile — top 20 domains:")
print(domain_sentiment.to_string())


Sentiment profile — top 20 domains:
                               mean  median  count    std
domain                                                   
yahoo.com                    -0.042  -0.021   2539  0.114
fool.com                     -0.018  -0.008   2330  0.242
benzinga.com                 -0.082  -0.004   1800  0.344
indiatimes.com               -0.078   0.004   1314  0.349
marketscreener.com           -0.043  -0.003   1232  0.312
finance.yahoo.com             0.024   0.028   1195  0.461
insidermonkey.com            -0.002  -0.002    907  0.316
webpronews.com               -0.052   0.005    827  0.347
themarketsdaily.com          -0.054  -0.003    684  0.300
markets.financialcontent.com  0.097   0.087    643  0.595
tickerreport.com             -0.047   0.005    621  0.303
moneycontrol.com             -0.070   0.002    551  0.381
cnbc.com                     -0.064  -0.021    534  0.319
forbes.com                   -0.041  -0.006    528  0.289
dailypolitical.com           -0.050 